# Public charging points per 1,000 households per municipality

This notebook maps the number of regular public charging points per 1,000 households in all Dutch municipalities.

**Required files:**
- `data/processed/chargingpoints_households_income.csv`: charging points (NAL, July 2026) and households/income (CBS 86161NED, 2024*) per municipality, created by notebook 1
- `data/raw/gemeenten_2024.gpkg`: municipal boundaries from CBS/PDOK, 2024 municipal division

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Load the data

The processed file contains the number of charging points and the number of households (in thousands) per municipality. We convert households to actual counts and calculate the number of charging points per 1,000 households.

In [ ]:
df = pd.read_csv("../data/processed/chargingpoints_households_income.csv")

# Households are given in thousands -> convert to actual counts
df["households"] = df["households_x1000"] * 1000

# Charging points per 1,000 households
df["chargingpoints_per_1000_households"] = df["charging_points"] / df["households"] * 1000

data = df[["municipality", "chargingpoints_per_1000_households"]].copy()
print("Number of municipalities:", len(data))
display(data.head())

## 2. Load municipal boundaries

We use the municipal boundaries from CBS/PDOK (2024 municipal division, generalised) from `data/raw/gemeenten_2024.gpkg`. If that file is missing, the boundaries are downloaded directly from PDOK.

In [ ]:
layer = "gemeente_gegeneraliseerd"

if os.path.exists("../data/raw/gemeenten_2024.gpkg"):
    municipalities = gpd.read_file("../data/raw/gemeenten_2024.gpkg", layer=layer)
else:
    url = ("https://service.pdok.nl/cbs/gebiedsindelingen/2024/wfs/v1_0"
           "?request=GetFeature&service=WFS&version=2.0.0"
           "&typeName=gemeente_gegeneraliseerd&outputFormat=json")
    municipalities = gpd.read_file(url)

print("Number of municipalities on the map:", len(municipalities))
display(municipalities.head())

## 3. Harmonise municipality names

CBS spells a few municipalities differently from the charging point file, for example *'s-Gravenhage* instead of *Den Haag*. These names are aligned so every municipality can be matched.

In [ ]:
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
municipalities["municipality"] = municipalities["statnaam"].replace(name_mapping)

# Check: both lines should print set()
print("Only on the map:", set(municipalities["municipality"]) - set(data["municipality"]))
print("Only in the data:", set(data["municipality"]) - set(municipalities["municipality"]))

## 4. Join the data to the map

In [ ]:
map_gdf = municipalities.merge(data, on="municipality", how="left")
print("Municipalities without data:", map_gdf["chargingpoints_per_1000_households"].isna().sum())

## 5. Create the map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

map_gdf.plot(
    column="chargingpoints_per_1000_households",
    cmap="Blues",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Charging points per 1,000 households", "shrink": 0.6},
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "No data"},
)

ax.set_title("Public charging points per 1,000 households, 2026", fontsize=16)
ax.axis("off")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/map_chargingpoints_per_1000households.png", dpi=200)
plt.show()